# 01 - Ingesta y preparación del dataset de consumo eléctrico

En este cuaderno se descarga el dataset **Individual Household Electric Power Consumption** desde el repositorio UCI mediante la librería `ucimlrepo`, se guarda una copia en formato CSV en la carpeta `data` y se realiza una primera exploración de la información disponible.

In [1]:
from pathlib import Path
from ucimlrepo import fetch_ucirepo 
import pandas as pd
from ucimlrepo import fetch_ucirepo

## Descarga del dataset desde UCI con `ucimlrepo`

In [3]:
individual_household_electric_power_consumption = fetch_ucirepo(id=235)

X = individual_household_electric_power_consumption.data.features
y = individual_household_electric_power_consumption.data.targets

print(individual_household_electric_power_consumption.metadata)
print(individual_household_electric_power_consumption.variables)

d:\Programs\Anaconda3\Lib\site-packages\ucimlrepo\fetch.py:97: DtypeWarning: Columns (2,3,4,5,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_url)


{'uci_id': 235, 'name': 'Individual Household Electric Power Consumption', 'repository_url': 'https://archive.ics.uci.edu/dataset/235/individual+household+electric+power+consumption', 'data_url': 'https://archive.ics.uci.edu/static/public/235/data.csv', 'abstract': 'Measurements of electric power consumption in one household with a one-minute sampling rate over a period of almost 4 years. Different electrical quantities and some sub-metering values are available.', 'area': 'Physics and Chemistry', 'tasks': ['Regression', 'Clustering'], 'characteristics': ['Multivariate', 'Time-Series'], 'num_instances': 2075259, 'num_features': 9, 'feature_types': ['Real'], 'demographics': [], 'target_col': None, 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2006, 'last_updated': 'Fri Mar 08 2024', 'dataset_doi': '10.24432/C58K54', 'creators': ['Georges Hebrail', 'Alice Berard'], 'intro_paper': None, 'additional_info': {'summary': 'This archiv

## Unión de datos y guardado en CSV dentro de la carpeta `data`

In [ ]:
df = pd.concat([X, y], axis=1)

data_dir = Path("..") / "data"
data_dir.mkdir(parents=True, exist_ok=True)

output_path = data_dir / "household_power_consumption.csv"
df.to_csv(output_path, index=False)

output_path

WindowsPath('../data/household_power_consumption.csv')

## Carga del CSV y exploración inicial

In [4]:
csv_path = Path("..") / "data" / "household_power_consumption.csv"
df = pd.read_csv(csv_path)

df.head()

C:\Users\frany\AppData\Local\Temp\ipykernel_17776\1359410438.py:2: DtypeWarning: Columns (2,3,4,5,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path)


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.840,18.400,0.000,1.000,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.630,23.000,0.000,1.000,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.290,23.000,0.000,2.000,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.740,23.000,0.000,1.000,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.680,15.800,0.000,1.000,17.0


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2075259 entries, 0 to 2075258
Data columns (total 9 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   Date                   object 
 1   Time                   object 
 2   Global_active_power    object 
 3   Global_reactive_power  object 
 4   Voltage                object 
 5   Global_intensity       object 
 6   Sub_metering_1         object 
 7   Sub_metering_2         object 
 8   Sub_metering_3         float64
dtypes: float64(1), object(8)
memory usage: 142.5+ MB


In [ ]:
# Ensure Global_active_power is stored as float
# Replace '?' with NaN and then cast to float
df["Global_active_power"] = df["Global_active_power"].replace("?", float("nan")).astype(float)

df["Global_active_power"] = df["Global_active_power"].astype(float)


In [8]:
# Verificar el contenido de todas las columnas
for col in df.columns:
    print(f"Columna: {col}")
    print(df[col].value_counts(dropna=False))
    print("-" * 40)


Columna: Date
Date
6/12/2008     1440
9/8/2009      1440
7/8/2009      1440
6/8/2009      1440
5/8/2009      1440
              ... 
5/4/2008      1440
4/4/2008      1440
24/11/2010    1440
26/11/2010    1263
16/12/2006     396
Name: count, Length: 1442, dtype: int64
----------------------------------------
Columna: Time
Time
17:24:00    1442
19:55:00    1442
19:44:00    1442
19:45:00    1442
19:46:00    1442
            ... 
03:47:00    1441
03:46:00    1441
03:45:00    1441
03:44:00    1441
17:23:00    1441
Name: count, Length: 1440, dtype: int64
----------------------------------------
Columna: Global_active_power
Global_active_power
NaN      25979
0.218     9565
0.216     9363
0.322     9350
0.324     9304
         ...  
7.094        1
7.930        1
8.274        1
8.246        1
8.600        1
Name: count, Length: 4187, dtype: int64
----------------------------------------
Columna: Global_reactive_power
Global_reactive_power
0.000    472786
?         25979
0.100     21577
0.102   

In [9]:
import pandas as pd

def asegurar_tipos_columnas(df):
    """
    Recorre todas las columnas del DataFrame y convierte cada una
    al tipo más apropiado: int, float o str según su contenido.
    """
    for col in df.columns:
        # Intentar convertir a numérico primero
        try:
            # Primero a float para no perder decimales
            df[col] = pd.to_numeric(df[col], errors='raise')
            # Si todos los valores son enteros, convertir a int
            if df[col].dropna().apply(lambda x: float(x).is_integer()).all():
                df[col] = df[col].astype('Int64')  # Int64 permite NaNs
        except (ValueError, TypeError):
            # Si no es numérico, dejar como string
            df[col] = df[col].astype(str)
    return df


In [10]:
df = asegurar_tipos_columnas(df)

In [11]:
df.head()

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.840,18.400,0.000,1.000,17
1,16/12/2006,17:25:00,5.360,0.436,233.630,23.000,0.000,1.000,16
2,16/12/2006,17:26:00,5.374,0.498,233.290,23.000,0.000,2.000,17
3,16/12/2006,17:27:00,5.388,0.502,233.740,23.000,0.000,1.000,17
4,16/12/2006,17:28:00,3.666,0.528,235.680,15.800,0.000,1.000,17


In [12]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2075259 entries, 0 to 2075258
Data columns (total 9 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   Date                   object 
 1   Time                   object 
 2   Global_active_power    float64
 3   Global_reactive_power  object 
 4   Voltage                object 
 5   Global_intensity       object 
 6   Sub_metering_1         object 
 7   Sub_metering_2         object 
 8   Sub_metering_3         Int64  
dtypes: Int64(1), float64(1), object(7)
memory usage: 144.5+ MB


In [15]:
data_dir = Path("..") / "data"
data_dir.mkdir(parents=True, exist_ok=True)

output_path = data_dir / "cleaned_energy_data.csv"
df.to_csv(output_path, index=False)
